# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates the loading and exploration of the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the `mlcroissant` library, which leverages the Croissant metadata schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Available record sets: {getattr(metadata, 'recordSet', []) if hasattr(metadata, 'recordSet') else []}")

## 2. Data Overview
List available record sets and for each, show its fields (columns) and their Croissant `@id` identifiers as defined in the schema.

In [ ]:
# List all record sets and their fields using @id references

# Get all record set @ids via metadata.record_sets (if present)
# When using mlcroissant, the available record sets can also be listed from the dataset object:

record_set_ids = dataset.record_set_ids
if not record_set_ids:
    print('No record sets detected in metadata. Trying to infer from files...')
else:
    print('Available record sets and fields:')
    for rs_id in record_set_ids:
        # Each record set has fields with @id's
        record_set = dataset.get_record_set(rs_id)
        print(f"- Record set @id: {rs_id}")
        fields = record_set.field_ids
        print(f"  Fields @ids: {fields}")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis.

> **Note:** For this dataset, Croissant metadata is authoritative. Always use the `@id`s to reference record sets and fields.

In [ ]:
# Extract data from all available record sets

dataframes = {}
if not record_set_ids:
    print('No record sets available for extraction.')
else:
    print('Extracting data from record sets:')
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame.from_records(records)
            dataframes[record_set_id] = df
            print(f"- Extracted {len(df)} records for record set @id: {record_set_id}")
        else:
            print(f"- No records found for record set @id: {record_set_id}")

if dataframes:
    # Select the first loaded record set for preview
    example_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns for record set @id '{example_record_set_id}':")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print('No dataframes created. Check dataset or record sets.')

## 4. Exploratory Data Analysis (EDA)
Perform exploratory analysis, such as filtering records, normalizing numeric fields, and grouping by key attributes.

> **Reminder:** Replace `<field_id>` variables with the actual `@id` values appropriate for your analysis. If unsure, check the column names printed above.

In [ ]:
import numpy as np

if dataframes:
    # Use the same example_record_set_id as above
    df = dataframes[example_record_set_id]
    print(f"Working with record set @id: {example_record_set_id}")
    
    # Infer a numeric field by type or naming (if present)
    numeric_candidate_cols = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    if numeric_candidate_cols:
        numeric_field_id = numeric_candidate_cols[0]
        print(f"Selected numeric field for analysis: {numeric_field_id}")
        threshold = df[numeric_field_id].quantile(0.9) if df[numeric_field_id].notnull().sum() > 10 else 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, normalized_col]].head())

        # Attempt to group by a likely categorical field if one exists
        categorical_cols = [col for col in df.select_dtypes(include=['object', 'category']).columns if col != numeric_field_id]
        group_field_id = categorical_cols[0] if categorical_cols else None
        if group_field_id:
            print(f"\nGrouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields found in this record set.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization

Visualize data distributions or field relationships using Matplotlib or other visualization libraries. If the dataset provides visualization metadata, you can also refer to its `visualization` section.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_candidate_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in record set @id {example_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(9, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df.dropna(subset=[group_field_id, numeric_field_id]))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Visualization skipped: no suitable numeric field or dataframe found.')

## 6. Conclusion

- We used the `mlcroissant` package to programmatically load and examine a dataset defined by a Croissant schema.
- All dataset elements (record sets, fields) were referenced consistently by their `@id` values.
- We performed data extraction, basic exploration, and demonstrated data visualization.

**Next steps:** You can extend this notebook for further analysis, modeling, or integrate it into data pipelines, always using Croissant schema semantics and `@id` referencing for robust and reproducible workflows.